# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [ ]:
import duckdb
from getpass import getpass

con = duckdb.connect()

# Token entered securely, not pasted in the cell — this repo is public
hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Same single-month partition as the Week 4 baseline — identical data slice
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

Paste your Hugging Face READ token: ··········


## 0.5 Rebuild inputs (label, features, client map)

*Everything below is rebuilt from the Week 4 baseline notebook so the model trains on the exact same rows and the exact same label. Nothing here is new logic — the only new logic starts in Section 1.*

In [ ]:
# Rebuild label_df — identical to w03/w04. Never touch this to build features.
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        impr_first_half,
        impr_second_half,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

print(f"Rows: {len(label_df)}")
print(f"Declining rate: {label_df['is_declining_proxy'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 151981
Declining rate: 0.438


In [ ]:
# Full-month position / click signal — identical to w04's pf / pf_valid
pf = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_full,
        SUM(gsc_impressions) AS total_impressions_full,
        SUM(gsc_clicks) AS total_clicks_full
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

pf_valid = pf.dropna(subset=["avg_position_full"]).copy()
pf_valid["eligible"] = pf_valid["total_impressions_full"] >= 10
pf_valid["zero_clicks_at_position"] = (
    (pf_valid["avg_position_full"] <= 10) & (pf_valid["total_clicks_full"] == 0) & (pf_valid["eligible"])
).astype(int)

print(pf_valid.shape)
print("eligible:", pf_valid['eligible'].sum(), "/ excluded:", (~pf_valid['eligible']).sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(175304, 6)
eligible: 143183 / excluded: 32121


In [ ]:
# First-half-only volume/click signal — same window discipline postrend already uses for
# position_change. These, not the full-month columns, are what the model trains on.
pf_fh = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_fh,
        SUM(gsc_impressions) AS total_impressions_fh,
        SUM(gsc_clicks) AS total_clicks_fh
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

pf_fh = pf_fh.dropna(subset=["avg_position_fh"]).copy()
pf_fh["ctr_fh"] = pf_fh["total_clicks_fh"] / pf_fh["total_impressions_fh"]

print(pf_fh.shape)

(150675, 5)


In [ ]:
# Leakage-safe position trend (days 1-15 only) — identical to w04's postrend / postrend_eligible,
# kept as a CONTINUOUS position_change here instead of collapsing straight to the binary flag.
postrend = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date < '2026-03-08' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk1,
        AVG(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk2,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

postrend = postrend.dropna(subset=["avg_position_wk1", "avg_position_wk2"])
postrend["position_change"] = postrend["avg_position_wk2"] - postrend["avg_position_wk1"]
postrend["position_worsened"] = (postrend["position_change"] > 0).astype(int)

# Same eligibility gate as w04, applied by EXCLUDING ineligible rows, not recoding them
postrend_eligible = postrend.merge(pf_valid[["content_hash_id", "eligible"]], on="content_hash_id", how="left")
postrend_eligible = postrend_eligible[postrend_eligible["eligible"] == True].copy()
print("eligible rows with a position trend:", len(postrend_eligible))

eligible rows with a position trend: 119176


In [ ]:
# Client map for the grouped split — one row per content_hash_id, never a feature
client_map = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('{month_path}')
""").df()

dupe_check = client_map.groupby("content_hash_id")["client_hash_id"].nunique()
assert (dupe_check == 1).all(), "content_hash_id maps to more than one client_hash_id"
print(f"{client_map['content_hash_id'].nunique()} pages across {client_map['client_hash_id'].nunique()} clients")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331437 pages across 55 clients


In [ ]:
import pandas as pd

# Assemble the model table. Left-join the position trend on (rows without one are real —
# not every page has 15 clean days of position data in the first half — flag it, don't hide it).
model_df = pf_fh.merge(
    pf_valid[["content_hash_id", "avg_position_full", "total_impressions_full",
              "total_clicks_full", "eligible", "zero_clicks_at_position"]],
    on="content_hash_id", how="inner"
)

model_df = model_df.merge(
    postrend_eligible[["content_hash_id", "position_change", "position_worsened"]],
    on="content_hash_id", how="left"
)
model_df["has_position_trend"] = model_df["position_change"].notna().astype(int)
model_df["position_change"] = model_df["position_change"].fillna(0)
model_df["position_worsened"] = model_df["position_worsened"].fillna(0).astype(int)

model_df = model_df.merge(client_map, on="content_hash_id", how="left")
print("rows missing a client mapping:", model_df["client_hash_id"].isna().sum())
model_df = model_df.dropna(subset=["client_hash_id"])

# Inner join to the label — same population the Week 4 baseline scored against (scored_labeled in w04)
model_df = model_df.merge(label_df[["content_hash_id", "is_declining_proxy"]], on="content_hash_id", how="inner")
model_df = model_df.sort_values("content_hash_id").reset_index(drop=True)

print(f"\nmodel_df: {model_df.shape}")
print(f"base rate: {model_df['is_declining_proxy'].mean():.3f}")
assert "impr_first_half" not in model_df.columns and "impr_second_half" not in model_df.columns

rows missing a client mapping: 0

model_df: (150675, 15)
base rate: 0.438


In [ ]:
import numpy as np

model_df["log_impressions_fh"] = np.log1p(model_df["total_impressions_fh"])
model_df["log_clicks_fh"] = np.log1p(model_df["total_clicks_fh"])

In [ ]:
# Leakage-safe version of the rule's AND-interaction, built the same way pf_valid's
# zero_clicks_at_position was, but from pf_fh (first-half-only) columns instead of full-month.
model_df["eligible_fh"] = model_df["total_impressions_fh"] >= 10
model_df["zero_clicks_at_position_fh"] = (
    (model_df["avg_position_fh"] <= 10)
    & (model_df["total_clicks_fh"] == 0)
    & (model_df["eligible_fh"])
).astype(int)

# The explicit AND the rule uses — position_worsened is already leakage-safe (from postrend)
model_df["zero_clicks_and_worsened_fh"] = (
    model_df["zero_clicks_at_position_fh"] * model_df["position_worsened"]
)

print("zero_clicks_at_position_fh rate:", model_df["zero_clicks_at_position_fh"].mean().round(3))
print("interaction rate:", model_df["zero_clicks_and_worsened_fh"].mean().round(3))

zero_clicks_at_position_fh rate: 0.199
interaction rate: 0.092


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

`is_declining_proxy` is a binary, observed label — not a proxy I'm building for the first time, the same one the baseline was checked against. Per the method menu that's Logistic Regression first, then Random Forest.

The reason I'm doing both is because my baseline rule *is* an interaction — `zero_clicks_at_position AND position_worsened` gets the top score, either alone gets a lower score. A logistic regression with these features as plain linear terms can't represent that AND without me hand-adding an interaction term. A tree finds it on its own by splitting on one feature, then the other, in the same branch. So LR tells me how much of the baseline's power is just "low position is bad, few clicks is bad" added together, and RF tells me whether the combination itself is doing real work beyond what's additive.

I'm scoring both the same way the baseline is scored — precision@K on a ranked list — because the deliverable is a queue, not a label. A model that nails accuracy but ranks badly at K=20 isn't useful here.

In [ ]:
feature_cols = ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh",
                "position_change", "has_position_trend"]

print("Features going into both models:")
for c in feature_cols:
    print(f"  {c:<20} {model_df[c].dtype}")

# engagement_weak / session_rate is deliberately excluded — Week 4's own signal check found it
# OPPOSITE-direction and confounded with impression volume. Not a safe input.
print("\nExcluded on purpose: session_rate / engagement_weak (confounded, wrong-direction in w04 signal check)")

Features going into both models:
  avg_position_fh      float64
  log_impressions_fh   float64
  log_clicks_fh        float64
  ctr_fh               float64
  position_change      float64
  has_position_trend   int64

Excluded on purpose: session_rate / engagement_weak (confounded, wrong-direction in w04 signal check)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by `client_hash_id`, single 80/20 split. Pages from the same client share templates, internal linking, content ops, and often the same underlying SEO problems — a random row split would let the model see a client's quirks in training and get credit for "predicting" a near-duplicate of that same client in test. That's not actually generalization to a new client, and the real use case is scoring clients broadly, including ones the model hasn't priced in yet.

Not time-aware: This is a single month (March 2026) with the label itself already built from a within-month time split (first half vs. second half), so there isn't a second, independent time axis left to split on without cutting into the label's own window.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = model_df[feature_cols].astype(float)
y = model_df["is_declining_proxy"].astype(int)
groups = model_df["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print(f"Train: {len(X_train):>6} rows, {groups_train.nunique():>3} clients, base rate {y_train.mean():.3f}")
print(f"Test:  {len(X_test):>6} rows, {groups_test.nunique():>3} clients, base rate {y_test.mean():.3f}")

overlap = set(groups_train) & set(groups_test)
print(f"\nClients present in both splits: {len(overlap)}")
assert len(overlap) == 0, "Group split leaked a client across train/test"

Train: 137554 rows,  35 clients, base rate 0.436
Test:   13121 rows,   9 clients, base rate 0.463

Clients present in both splits: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The frozen Week 4 numbers (precision@20/50/100/200 = 0.600 / 0.560 / 0.510 / 0.505) were computed over the *entire* population, since the rule had nothing to fit and no test set. That's not a fair number to hold the model to as the model only gets credit for rows it never trained on, so the Week 4 baseline needs to be re-scored on that same held-out test split for the comparison to mean anything. These recomputed baseline numbers can be expected to drift a little from the frozen ones purely from being a subset.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def precision_at_k(order_labels, k):
    return np.asarray(order_labels)[:k].mean()

RANDOM_STATE = 42

# Logistic Regression — scaled, since coefficients on avg_position vs total_impressions
# aren't on comparable scales otherwise
lr = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
lr.fit(X_train, y_train)
lr_scores = lr.predict_proba(X_test)[:, 1]

# Random Forest — shallow-ish and leaf-floored on purpose: this is ~tens of thousands of rows,
# not millions, and a depth-2-readable baseline is the bar to clear, not a black box
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print("Models fit.")

Models fit.


In [ ]:
# Re-score the FROZEN Week 4 rule on the test rows only — same score formula, same tie-break
# (score, then total_impressions) as w04's own ranking, just restricted to the test split.
baseline_test = model_df.loc[X_test.index].copy()
baseline_test["score"] = baseline_test["zero_clicks_at_position"] * 2 + baseline_test["position_worsened"] * 1
baseline_order = baseline_test.sort_values(["score", "total_impressions_full"], ascending=[False, False]).index
baseline_ranked_labels = y_test.loc[baseline_order].values

# LR / RF: continuous scores, ties are rare enough that argsort's default order is fine
lr_order = np.argsort(-lr_scores)
rf_order = np.argsort(-rf_scores)
lr_ranked_labels = y_test.values[lr_order]
rf_ranked_labels = y_test.values[rf_order]

ks = [20, 50, 100, 200]
rows = []
for k in ks:
    rows.append({
        "k": k,
        "base_rate": round(y_test.mean(), 3),
        "baseline_rule": round(precision_at_k(baseline_ranked_labels, k), 3),
        "logistic_regression": round(precision_at_k(lr_ranked_labels, k), 3),
        "random_forest": round(precision_at_k(rf_ranked_labels, k), 3),
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

print(f"\n(Reference — Week 4 frozen baseline on the FULL population: "
      f"P@20=0.600 P@50=0.560 P@100=0.510 P@200=0.505)")

  k  base_rate  baseline_rule  logistic_regression  random_forest
 20      0.463           0.70                 0.30          0.600
 50      0.463           0.60                 0.20          0.580
100      0.463           0.51                 0.27          0.550
200      0.463           0.51                 0.33          0.625

(Reference — Week 4 frozen baseline on the FULL population: P@20=0.600 P@50=0.560 P@100=0.510 P@200=0.505)


In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr.named_steps["clf"].coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df.to_string(index=False))

           feature  coefficient
     log_clicks_fh    -0.279124
log_impressions_fh     0.245463
   avg_position_fh    -0.130718
has_position_trend    -0.112243
   position_change     0.057509
            ctr_fh     0.028959


The two models split from the baseline rule in different ways — RF is mixed by K, LR loses outright.

- **RF vs rule:** at K=20 and K=50, the rule wins (0.70 and 0.60 vs. RF's 0.60 and 0.58). At K=100 and K=200, RF wins (0.55 vs. 0.51, 0.625 vs. 0.51). The rule was hand-built to front-load its strongest cases, and that's exactly where it holds up; past the top ~50 it runs out of hand-coded signal, while RF keeps finding real cases the rule's binary AND-gate never touches at all (see the rule-missed rows in Section 4).
- **LR vs rule:** LR loses at every K, and not narrowly — 0.30/0.20/0.27/0.33, all below the 0.463 base rate. It's actually worse than picking rows at random, not just weaker than the rule. It only catches up once the rule's own interaction is handed to it explicitly as a feature (Section 4 diagnostic) — unassisted, as actually submitted here, it isn't usable for this task.

Short queue: use the rule. Longer queue: RF earns its keep. LR as built here doesn't win at any queue length without the interaction feature it's missing.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="average_precision"
)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print(importance_df.to_string(index=False))

           feature  importance_mean  importance_std
log_impressions_fh         0.023081        0.002023
            ctr_fh         0.022130        0.002087
   avg_position_fh         0.014533        0.003178
has_position_trend         0.005583        0.002175
     log_clicks_fh        -0.001574        0.001142
   position_change        -0.007150        0.002756


In [ ]:
# Wrong picks near the top of the RF-ranked queue — same spirit as the w04 top-10 review
test_results = baseline_test.copy()
test_results["y_true"] = y_test.values
test_results["rf_score"] = rf_scores
test_results["lr_score"] = lr_scores

rf_top50 = test_results.sort_values("rf_score", ascending=False).head(50)
false_positives = rf_top50[rf_top50["y_true"] == 0]
print(f"RF top-50: {len(false_positives)} false positives (flagged high, not actually declining)\n")
print(false_positives[["content_hash_id", "avg_position_fh", "total_impressions_fh",
                        "total_clicks_fh", "position_change", "rf_score"]].head(10).to_string(index=False))

RF top-50: 21 false positives (flagged high, not actually declining)

         content_hash_id  avg_position_fh  total_impressions_fh  total_clicks_fh  position_change  rf_score
content_2dc954b9ae28df53        55.996136                6951.0              8.0        10.930600  0.757650
content_39457d17e716086c        38.271078               25042.0              7.0         0.614578  0.739078
content_9a4594adab0f2c81        34.689023               14995.0             11.0        11.346807  0.721109
content_567d370cf1fdbd1d        38.718105               12318.0              8.0        -3.985967  0.717614
content_d1db17521a55d9fc        30.665879               19059.0             19.0        -0.078077  0.712756
content_beaed0a00aacabe4        32.615764                7986.0             15.0         3.953476  0.703164
content_6be250b9619dc292        29.973380                5415.0              4.0         6.505907  0.687387
content_e7447675215180c2        29.995171                6925.0   

In [ ]:
# Where the model and the baseline rule DISAGREE most — the top-50 RF picks the rule missed entirely
rule_missed = rf_top50[rf_top50["score"] == 0]
print(f"RF top-50 rows the baseline rule scored 0 (no flag at all): {len(rule_missed)}")
print(rule_missed[["content_hash_id", "avg_position_fh", "total_impressions_fh",
                    "total_clicks_fh", "ctr_fh", "position_change", "rf_score", "y_true"]].head(10).to_string(index=False))

RF top-50 rows the baseline rule scored 0 (no flag at all): 13
         content_hash_id  avg_position_fh  total_impressions_fh  total_clicks_fh   ctr_fh  position_change  rf_score  y_true
content_44c1697a4863f94f        46.060050                5065.0              5.0 0.000987        -1.511370  0.718767       1
content_567d370cf1fdbd1d        38.718105               12318.0              8.0 0.000649        -3.985967  0.717614       0
content_459450adc5699701        43.093670               16532.0             24.0 0.001452        -4.081878  0.713837       1
content_d1db17521a55d9fc        30.665879               19059.0             19.0 0.000997        -0.078077  0.712756       0
content_ba4ed4a2c2f270bc        32.955850                3586.0              4.0 0.001115        -3.068615  0.664928       1
content_a4ef56694f95b3e9        35.664002                3178.0              8.0 0.002517        -3.322801  0.639606       1
content_5170178abf80b0f8        27.856809                6726.

In [ ]:
# How much of LR's top-K is just "rows where the interaction flag fired"?
lr_top20 = test_results.sort_values("lr_score", ascending=False).head(20)
lr_top50 = test_results.sort_values("lr_score", ascending=False).head(50)

overall_rate = model_df.loc[X_test.index, "zero_clicks_and_worsened_fh"].mean()
print("Interaction flag prevalence, test set overall:", round(overall_rate, 3))
print("Interaction flag rate in LR top-20:", round(lr_top20["zero_clicks_and_worsened_fh"].mean(), 3))
print("Interaction flag rate in LR top-50:", round(lr_top50["zero_clicks_and_worsened_fh"].mean(), 3))

Interaction flag prevalence, test set overall: 0.094
Interaction flag rate in LR top-20: 0.05
Interaction flag rate in LR top-50: 0.06


### Why does LR underperform at low K?

Without an interaction feature, LR's top-K picks contain rows where
`zero_clicks_at_position AND position_worsened` holds at only 5-6% — *below*
the 9.4% base rate, meaning LR isn't just blind to the interaction, it mildly
deprioritizes it. As a diagnostic — not a competing final model — I refit LR
with that interaction handed to it explicitly, to check whether that alone
explains the gap.

In [21]:
# eligible_fh, zero_clicks_at_position_fh, and zero_clicks_and_worsened_fh were already
# built on model_df in cell 11 — reused here, not recomputed.
diag_feature_cols = feature_cols + ["zero_clicks_and_worsened_fh"]
X_train_diag = model_df.loc[X_train.index, diag_feature_cols].astype(float)
X_test_diag = model_df.loc[X_test.index, diag_feature_cols].astype(float)

lr_diag = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
lr_diag.fit(X_train_diag, y_train)
lr_diag_scores = lr_diag.predict_proba(X_test_diag)[:, 1]

lr_diag_order = np.argsort(-lr_diag_scores)
lr_diag_ranked_labels = y_test.values[lr_diag_order]

print(f"{'k':>5} {'lr_no_interaction':>18} {'lr_with_interaction':>20}")
for k in [20, 50, 100, 200]:
    print(f"{k:>5} {precision_at_k(lr_ranked_labels, k):>18.3f} {precision_at_k(lr_diag_ranked_labels, k):>20.3f}")

    k  lr_no_interaction  lr_with_interaction
   20              0.300                0.900
   50              0.200                0.700
  100              0.270                0.630
  200              0.330                0.635


In [ ]:
diag_results = baseline_test.copy()
diag_results["y_true"] = y_test.values
diag_results["lr_diag_score"] = lr_diag_scores
diag_results["zero_clicks_and_worsened_fh"] = model_df.loc[X_test.index, "zero_clicks_and_worsened_fh"].values

diag_top20 = diag_results.sort_values("lr_diag_score", ascending=False).head(20)
diag_top50 = diag_results.sort_values("lr_diag_score", ascending=False).head(50)

print("Interaction flag prevalence, test set overall:",
      round(diag_results["zero_clicks_and_worsened_fh"].mean(), 3))
print("Interaction flag rate in diagnostic LR top-20:", round(diag_top20["zero_clicks_and_worsened_fh"].mean(), 3))
print("Interaction flag rate in diagnostic LR top-50:", round(diag_top50["zero_clicks_and_worsened_fh"].mean(), 3))

Interaction flag prevalence, test set overall: 0.094
Interaction flag rate in diagnostic LR top-20: 0.95
Interaction flag rate in diagnostic LR top-50: 0.84


**Top permutation-importance feature:**
- Nominally `log_impressions_fh` (0.023) is the top feature, but it's within one standard deviation of `ctr_fh` (0.022) — effectively tied, not a clean winner. Neither feature maps cleanly onto Week 4's `zero_clicks_at_position`, which was itself an interaction between position and clicks.
- Permutation importance shuffles one feature at a time, so a feature that only matters in combination with another can look weak alone even while RF leans on it heavily inside a joint split — `avg_position_fh` ranks third (0.015) and `log_clicks_fh` comes back essentially at zero (-0.002), consistent with that: position and clicks aren't unimportant, they're just not separable from each other in this view.

**False positives (cell 23):**
- False positives don't match the Week 4 tracking-artifact pattern (page-one position, real volume, literally zero clicks). Every one has `avg_position_fh` in the high-20s to mid-50s — well outside the rule's page-one cutoff — nonzero clicks, and several have large `position_change` (content_2dc954b9..., +10.9; content_9a4594ad..., +11.3).
- This looks like RF over-trusting a big position swing on a page that was never ranking well to begin with, where a 10-spot move matters far less to actual clicks than the same move would on page one. That's genuinely a new failure mode the rule didn't have, since the rule only checked the sign of the position change, not its size.

**Rule-missed rows RF flagged (cell 24):**
- Rule-missed rows are mixed, and a bit messy. Of the 10 shown, 4 are true positives and 6 are false. All share negative `position_change` (position actually improved) and moderate click counts, which is why the rule scored them 0 — neither `zero_clicks_at_position` nor `position_worsened` fires. Some of the true positives have plausibly low `ctr_fh` at real volume (0.0006–0.0033), which is a nameable reason. But at close to 50/50, it reads as a weak pattern that sometimes pays off instead of a reliable one.

**Top 3 features, why they plausibly matter:**
  1. `log_impressions_fh` — raw search demand for the page; where it sits in the volume distribution shapes how much room is left to decline.
  2. `ctr_fh` — impressions converting to clicks; a low ratio at a given position is the continuous version of the same "getting seen but not chosen" pattern the rule's zero-click signal was built to catch.
  3. `avg_position_fh` — ranking; at a fixed CTR, worse position caps how many clicks are even possible, so it's a natural companion to CTR rather than a fully separate signal.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.